##CS7DS2 - Optimisation Algorithms for Data Analysis - Week 8 Assignment

Name : Swetha Sekar

Student ID : 25336453

In [2]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from scipy.optimize import linprog

In [3]:
np.random.seed(42)

##Question - 1

$X_1 = \{0.5 \leq x_1 \leq 2.5,\; 0.5 \leq x_2 \leq 3.5\}$ is a box constraint. The primary idea behind the projected gradient descent is to take a normal gradient step to get an intermediate point $z_{t+1} = x_t - \alpha \nabla f(x_t)$ and then, if that point has wandered outside the feasible set, snap it back to the nearest point inside:
xt+1 = PX1 (zt+1)
Since $X_1$ has independent bounds per coordinate, the projection reduces
to elementwise clipping and there is no solver needed. The assignment requires at least 60 iterations, but here 80 iterations are used to give the convergence curve room to fully flatten out with step size $\alpha = 0.1$,
starting from $x^{(0)} = (2.4,\, 0.7)$.

In [5]:
def func_q1(x1, x2):
    return (x1 - 1.2)**2 + 2*(x2 - 2.5)**2 + 0.4*x1*x2

def grad_q1(x1, x2):
    df1 = 2*(x1 - 1.2) + 0.4*x2
    df2 = 4*(x2 - 2.5) + 0.4*x1
    return np.array([df1, df2])

def proj_X1(x):
    return np.clip(x, [0.5, 0.5], [2.5, 3.5])

def proj_X2(x):
    x = x.copy()
    x[0] = max(x[0], 0.5)
    x[1] = max(x[1], 0.5)
    if x[0] > x[1]:
        mid = (x[0] + x[1]) / 2.0
        x[0] = mid
        x[1] = mid
    x[0] = max(x[0], 0.5)
    x[1] = max(x[1], 0.5)
    return x

def pgd(x0, grad_fn, proj_fn, alpha, itr):
    x = x0.copy().astype(float)
    hist = [x.copy()]
    for _ in range(itr):
        g = grad_fn(*x)
        x = proj_fn(x - alpha * g)
        hist.append(x.copy())
    return np.array(hist)

In [6]:
x0_q1 = np.array([2.4, 0.7])
alpha_q1 = 0.1
n_q1 = 80

hist_X1 = pgd(x0_q1, grad_q1, proj_X1, alpha_q1, n_q1)
hist_X2 = pgd(x0_q1, grad_q1, proj_X2, alpha_q1, n_q1)

func_hist_X1 = np.array([func_q1(*p) for p in hist_X1])
func_hist_X2 = np.array([func_q1(*p) for p in hist_X2])

x1g = np.linspace(0.0, 3.0, 300)
x2g = np.linspace(0.0, 4.0, 300)
X1G, X2G = np.meshgrid(x1g, x2g)
ZQ1 = func_q1(X1G, X2G)

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
feas = plt.matplotlib.patches.Rectangle((0.5, 0.5), 2.0, 3.0, edgecolor=CBND, facecolor=CFEAS, zorder=0, label='Feasible $X_1$')
ax.add_patch(feas)
ax.contour(X1G, X2G, ZQ1, levels=20, zorder=1)
ax.plot(hist_X1[:,0], hist_X1[:,1], '-o', ms=3, zorder=2, label='Trajectory')
ax.plot(*x0_q1, 's', ms=8, label='$x^{(0)}$')
ax.plot(*hist_X1[-1], 'o', ms=12, label='Final iterate')
ax.set_xlim(0.0, 3.0); ax.set_ylim(0.0, 4.0)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Q1: (d-I) Contour + Trajectory – $X_1$')
ax.legend(fontsize=8)

ax = axes[1]
ax.semilogy(func_hist_X1, label='$f(x^{(t)})$')
ax.set_xlabel('Iteration'); ax.set_ylabel('$f(x^{(t)})$ [log scale]')
ax.set_title('(Q1: d-II) Loss vs Iteration – $X_1$')
ax.grid(True, alpha=0.3); ax.legend()

ax = axes[2]
ax.plot(hist_X1[:,0], lw=1.6, label='$x_1^{(t)}$')
ax.plot(hist_X1[:,1], lw=1.6, linestyle='--', label='$x_2^{(t)}$')
ax.set_xlabel('Iteration'); ax.set_ylabel('Coordinate value')
ax.set_title('Q1: (d-III) Coordinates vs Iteration – $X_1$')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('q1d_X1.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

In [16]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

from matplotlib.patches import Polygon as MplPolygon
verts_X2 = np.array([[0.5,0.5],[3.0,3.0],[0.5,3.0]])
ax = axes[0]
feas2 = MplPolygon(verts_X2, closed=True, zorder=0, edgecolor=CBND, facecolor=CFEAS, label='Feasible $X_2$')
ax.add_patch(feas2)
ax.contour(X1G, X2G, ZQ1, levels=20, zorder=1)
ax.plot(hist_X2[:,0], hist_X2[:,1], '-o', color=C1, ms=3, zorder=2, label='Trajectory')
ax.plot(*x0_q1, 's',  ms=8, label='$x^{(0)}$')
ax.plot(*hist_X2[-1], 'o', ms=12, label='Final iterate')
ax.set_xlim(0.0, 3.0); ax.set_ylim(0.0, 4.0)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Q1: (e-I) Contour + Trajectory – $X_2$')
ax.legend(fontsize=8)

ax = axes[1]
ax.semilogy(func_hist_X2, lw=1.8, label='$f(x^{(t)})$')
ax.set_xlabel('Iteration'); ax.set_ylabel('$f(x^{(t)})$ [log scale]')
ax.set_title('Q1: (e-II) Loss vs Iteration – $X_2$')
ax.grid(True, alpha=0.3); ax.legend()

ax = axes[2]
ax.plot(hist_X2[:,0], lw=1.6, label='$x_1^{(t)}$')
ax.plot(hist_X2[:,1], lw=1.6, linestyle='--', label='$x_2^{(t)}$')
ax.set_xlabel('Iteration'); ax.set_ylabel('Coordinate value')
ax.set_title('Q1: (e-III) Coordinates vs Iteration – $X_2$')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('q1e_X2.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

##Question 2

The penalty term is based on the positive parts of the constraint violations. The penalised objective is the
original objective plus a penalty weight times the sum of violations. A small penalty weight does not force
feasibility strongly enough. A larger penalty weight makes the solution move closer to the feasible region.


The primal-dual method jointly updates both the decision variables x (primal) and the Lagrange multipliers
λ (dual) at each iteration, making it more adaptive than the fixed penalty approach. Rather than fixing
λ upfront, the multipliers evolve throughout the run, growing when constraints are violated and stabilising
as the iterates approach feasibility.


In [19]:
def func_q2(x1, x2):
    return (x1 - 0.2)**2 + (x2 - 2)**2

def grad_q2(x1, x2):
    return np.array([2*(x1 - 0.2), 2*(x2 - 2)])

def g1(x1, x2): return 0.5 - x1
def g2(x1, x2): return 1.0 - x1*x2

def grad_g1(x1, x2): return np.array([-1.0, 0.0])
def grad_g2(x1, x2): return np.array([-x2, -x1])

def F_penalty(x1, x2, lam1, lam2):
    return func_q2(x1,x2) + lam1*max(0,g1(x1,x2)) + lam2*max(0,g2(x1,x2))

def compute_grad_F(x1, x2, lam1, lam2):
    gf = grad_q2(x1, x2)
    gp = np.zeros(2)
    if g1(x1,x2) > 0:
        gp += lam1 * grad_g1(x1,x2)
    if g2(x1,x2) > 0:
        gp += lam2 * grad_g2(x1,x2)
    return gf + gp

x0_q2 = np.array([1.4, 0.6])
alpha_q2 = 0.01
n_q2 = 300

In [20]:
def compute_gd_penalty_q2(x0, lam1, lam2, alpha, itr):
    x = x0.copy().astype(float)
    hist = [x.copy()]
    g1_hist, g2_hist = [g1(*x)], [g2(*x)]
    for _ in range(itr):
        g = compute_grad_F(*x, lam1, lam2)
        x = x - alpha * g
        hist.append(x.copy())
        g1_hist.append(g1(*x))
        g2_hist.append(g2(*x))
    return np.array(hist), np.array(g1_hist), np.array(g2_hist)

hist_small_q2, g1_small, g2_small = compute_gd_penalty_q2(x0_q2, 0.5, 0.5, alpha_q2, n_q2)
hist_large_q2, g1_large, g2_large = compute_gd_penalty_q2(x0_q2, 4.0, 4.0, alpha_q2, n_q2)

def primal_dual(x0, alpha=0.06, beta=0.08, n_iters=150):
    x = x0.copy().astype(float)
    lam = np.array([0.0, 0.0])
    hist_x = [x.copy()]
    hist_lam = [lam.copy()]
    for _ in range(n_iters):
        gf = grad_q2(*x)
        gcons = lam[0]*grad_g1(*x) + lam[1]*grad_g2(*x)
        x = x - alpha*(gf + gcons)
        lam[0] = max(0, lam[0] + beta*g1(*x))
        lam[1] = max(0, lam[1] + beta*g2(*x))
        hist_x.append(x.copy())
        hist_lam.append(lam.copy())
    return np.array(hist_x), np.array(hist_lam)

hist_pd, hist_lam = primal_dual(x0_q2, alpha=0.06, beta=0.08, n_iters=150)

x1g2 = np.linspace(0.1, 2.5, 300)
x2g2 = np.linspace(0.1, 3.5, 300)
X1G2, X2G2 = np.meshgrid(x1g2, x2g2)
ZQ2 = func_q2(X1G2, X2G2)

feas_mask = (X1G2 >= 0.5) & (X1G2*X2G2 >= 1.0)

def plot_feasible_q2(ax):
    ax.contourf(X1G2, X2G2, feas_mask.astype(float), levels=[0.5,1.5],
                colors=[CFEAS], alpha=0.6, zorder=0)
    xx = np.linspace(0.5, 2.5, 300)
    ax.plot(xx, 1.0/xx, color=CBND, lw=1.2, zorder=1, label='$x_1 x_2=1$')
    ax.axvline(0.5, color=CBND, lw=1.2, linestyle='--', label='$x_1=0.5$')

In [21]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
plot_feasible_q2(ax)
ax.contour(X1G2, X2G2, ZQ2, levels=15, zorder=1)
ax.plot(hist_small_q2[:,0], hist_small_q2[:,1], '-o', ms=2, label='Trajectory (small $\lambda$)')
ax.plot(*x0_q2, 's', ms=8, label='$x^{(0)}$')
ax.set_xlim(0.1,2.5); ax.set_ylim(0.1,3.5)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Q2: (d-I) Small penalty ($\lambda=0.5$)')
ax.legend(fontsize=8)

ax = axes[1]
plot_feasible_q2(ax)
ax.contour(X1G2, X2G2, ZQ2, levels=15, linewidths=0.7, zorder=1)
ax.plot(hist_large_q2[:,0], hist_large_q2[:,1], '-o', ms=2, lw=1.3, label='Trajectory (large $\lambda$)')
ax.plot(*x0_q2, 's', ms=8, label='$x^{(0)}$')
ax.set_xlim(0.1,2.5); ax.set_ylim(0.1,3.5)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Q2: (d-II) Large penalty ($\lambda=4.0$)')
ax.legend(fontsize=8)

ax = axes[2]
itr_q2 = np.arange(n_q2+1)
ax.plot(itr_q2, g1_small, lw=1.6, label='$g_1$ small $\lambda$')
ax.plot(itr_q2, g2_small, lw=1.6, linestyle='--', label='$g_2$ small $\lambda$')
ax.plot(itr_q2, g1_large, lw=1.6, label='$g_1$ large $\lambda$')
ax.plot(itr_q2, g2_large, lw=1.6, linestyle='--', label='$g_2$ large $\lambda$')
ax.axhline(0, color='#888888', lw=0.8, linestyle=':')
ax.set_xlabel('Iteration'); ax.set_ylabel('Constraint value')
ax.set_title('Q2: (d-III) Constraint violations vs Iteration')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('q2d_penalty.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

<>:6: SyntaxWarning: invalid escape sequence '\l'
<>:10: SyntaxWarning: invalid escape sequence '\l'
<>:16: SyntaxWarning: invalid escape sequence '\l'
<>:20: SyntaxWarning: invalid escape sequence '\l'
<>:25: SyntaxWarning: invalid escape sequence '\l'
<>:26: SyntaxWarning: invalid escape sequence '\l'
<>:27: SyntaxWarning: invalid escape sequence '\l'
<>:28: SyntaxWarning: invalid escape sequence '\l'
<>:6: SyntaxWarning: invalid escape sequence '\l'
<>:10: SyntaxWarning: invalid escape sequence '\l'
<>:16: SyntaxWarning: invalid escape sequence '\l'
<>:20: SyntaxWarning: invalid escape sequence '\l'
<>:25: SyntaxWarning: invalid escape sequence '\l'
<>:26: SyntaxWarning: invalid escape sequence '\l'
<>:27: SyntaxWarning: invalid escape sequence '\l'
<>:28: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_8903/3326618789.py:6: SyntaxWarning: invalid escape sequence '\l'
  ax.plot(hist_small_q2[:,0], hist_small_q2[:,1], '-o', ms=2, label='Trajectory (small $\lambda$)')
/tmp/

In [22]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
plot_feasible_q2(ax)
ax.contour(X1G2, X2G2, ZQ2, levels=15, zorder=1)
ax.plot(hist_pd[:,0], hist_pd[:,1], '-o', ms=2, label='Primal-dual trajectory')
ax.plot(*x0_q2, 's', ms=8, label='$x^{(0)}$')
ax.set_xlim(0.1,2.5); ax.set_ylim(0.1,3.5)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Q2: (e-I) Primal-dual contour + trajectory')
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(hist_pd[:,0], label='$x_1^{(t)}$')
ax.plot(hist_pd[:,1], linestyle='--', label='$x_2^{(t)}$')
ax.set_xlabel('Iteration'); ax.set_ylabel('Coordinate')
ax.set_title('Q2: (e-II) $x_1, x_2$ vs Iteration – Primal-dual')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(hist_lam[:,0], label='$\lambda_1^{(t)}$')
ax.plot(hist_lam[:,1], linestyle='--', label='$\lambda_2^{(t)}$')
ax.set_xlabel('Iteration'); ax.set_ylabel('Multiplier value')
ax.set_title('Q2: (e-III) $\lambda_1, \lambda_2$ vs Iteration')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('q2e_primaldual.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

<>:21: SyntaxWarning: invalid escape sequence '\l'
<>:22: SyntaxWarning: invalid escape sequence '\l'
<>:24: SyntaxWarning: invalid escape sequence '\l'
<>:21: SyntaxWarning: invalid escape sequence '\l'
<>:22: SyntaxWarning: invalid escape sequence '\l'
<>:24: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_8903/2931827522.py:21: SyntaxWarning: invalid escape sequence '\l'
  ax.plot(hist_lam[:,0], label='$\lambda_1^{(t)}$')
/tmp/ipykernel_8903/2931827522.py:22: SyntaxWarning: invalid escape sequence '\l'
  ax.plot(hist_lam[:,1], linestyle='--', label='$\lambda_2^{(t)}$')
/tmp/ipykernel_8903/2931827522.py:24: SyntaxWarning: invalid escape sequence '\l'
  ax.set_title('Q2: (e-III) $\lambda_1, \lambda_2$ vs Iteration')


##Question - 3

The Frank-Wolfe subproblem at each iteration is minz∈X ∇f (x(t))⊤z, which is a linear programme over a
polytope. By the fundamental theorem of linear programming, a linear objective over a bounded convex
polytope always attains its minimum at an extreme point (vertex). This follows because a linear function
has no curvature and therefore no interior stationary points, also the minimum must lie on the boundary
and on a polytope the extreme boundary points are exactly the vertices. In practice this means checking all
five vertices and picking the one that gives the smallest inner product with the current gradient, O(5) per
iteration with no projection required.

In [23]:
def f_q3(x1, x2):
    return (x1 - 1.5)**2 + (x2 - 1.2)**2

def grad_q3(x1, x2):
    return np.array([2*(x1 - 1.5), 2*(x2 - 1.2)])

VERTICES = np.array([
    [0.5, 0.5],
    [3.0, 0.5],
    [3.0, 1.0],   # x1=3, x1+x2=4 => x2=1
    [1.0, 3.0],   # x2=3, x1+x2=4 => x1=1
    [0.5, 3.0],
])

def frank_wolfe_lp(grad, vertices):
    scores = vertices @ grad
    return vertices[np.argmin(scores)]

def calc_frank_wolfe(x0, grad_fn, vertices, beta, itrs):
    x = x0.copy().astype(float)
    hist_x = [x.copy()]
    hist_z = []
    hist_f = [f_q3(*x)]
    for t in range(itrs):
        g = grad_fn(*x)
        z = frank_wolfe_lp(g, vertices)
        hist_z.append(z.copy())
        gamma = 2.0 / (t + 2)
        x = (1 - beta)*x + beta*z
        hist_x.append(x.copy())
        hist_f.append(f_q3(*x))
    return np.array(hist_x), np.array(hist_f), np.array(hist_z)

In [24]:
x0_q3 = np.array([2.8, 0.8])
n_q3 = 80

hist_fw08_q3, fv_fw08_q3, hz_fw08_q3 = calc_frank_wolfe(x0_q3, grad_q3, VERTICES, beta=0.8, itrs=n_q3)
hist_fw095_q3, fv_fw095_q3, hz_fw095_q3 = calc_frank_wolfe(x0_q3, grad_q3, VERTICES, beta=0.95, itrs=n_q3)

def project_X3(x):
    for _ in range(50):
        x[0] = np.clip(x[0], 0.5, 3.0)
        x[1] = np.clip(x[1], 0.5, 3.0)
        if x[0] + x[1] > 4.0:
            excess = x[0] + x[1] - 4.0
            x[0] -= excess/2; x[1] -= excess/2
    return x

hist_pgd3 = pgd(x0_q3, grad_q3, project_X3, alpha=0.05, itr=n_q3)
fv_pgd3 = np.array([f_q3(*p) for p in hist_pgd3])

x1g3 = np.linspace(0.0, 3.5, 300)
x2g3 = np.linspace(0.0, 3.5, 300)
X1G3, X2G3 = np.meshgrid(x1g3, x2g3)
ZQ3 = f_q3(X1G3, X2G3)
feas3 = (X1G3>=0.5)&(X2G3>=0.5)&(X1G3+X2G3<=4)&(X1G3<=3)&(X2G3<=3)

def draw_feasible_X3(ax):
    ax.contourf(X1G3, X2G3, feas3.astype(float), levels=[0.5,1.5],
                colors=[CFEAS], alpha=0.5, zorder=0)
    vx = np.append(VERTICES[:,0], VERTICES[0,0])
    vy = np.append(VERTICES[:,1], VERTICES[0,1])
    ax.plot(vx, vy, color=CBND, lw=1.2, zorder=1)
    ax.scatter(VERTICES[:,0], VERTICES[:,1], color=C4, s=50, zorder=3, label='Vertices')

In [25]:
for bval, hist_fw, fv_fw, hz_fw, tag in [
        (0.8, hist_fw08_q3, fv_fw08_q3, hz_fw08_q3, '08'),
        (0.95, hist_fw095_q3, fv_fw095_q3, hz_fw095_q3, '095')]:

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    ax = axes[0]
    draw_feasible_X3(ax)
    ax.contour(X1G3, X2G3, ZQ3, levels=15, zorder=1)
    ax.plot(hist_fw[:,0], hist_fw[:,1], '-o', ms=3, label='FW trajectory')
    ax.plot(*x0_q3, 's', ms=8, label='$x^{(0)}$')
    ax.plot(*hist_fw[-1], '*', ms=12, label='Final')
    ax.set_xlim(0,3.5); ax.set_ylim(0,3.5)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_title(f'Q3: (d-I) FW Contour + Trajectory ($\\beta={bval}$)')
    ax.legend(fontsize=8)

    ax = axes[1]
    ax.semilogy(fv_fw, label='$f(x^{(t)})$')
    ax.set_xlabel('Iteration'); ax.set_ylabel('$f$ [log]')
    ax.set_title(f'Q3: (d-II) $f(x^{{(t)}})$ vs Iteration ($\\beta={bval}$)')
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(hist_fw[:,0], label='$x_1^{(t)}$')
    ax.plot(hist_fw[:,1], linestyle='--', label='$x_2^{(t)}$')
    if len(hz_fw) > 0:
        ax.plot(np.arange(1, len(hz_fw)+1), hz_fw[:,0], linestyle=':', label='$z_1^{(t)}$')
        ax.plot(np.arange(1, len(hz_fw)+1), hz_fw[:,1], linestyle='-.', label='$z_2^{(t)}$')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Coordinate')
    ax.set_title(f'Q3: (d-III) $x^{{(t)}}$ and $z^{{(t)}}$ ($\\beta={bval}$)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'q3d_fw{tag}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

In [26]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.semilogy(fv_fw08_q3, label='FW $\\beta=0.8$')
ax.semilogy(fv_fw095_q3, linestyle='--', label='FW $\\beta=0.95$')
ax.semilogy(fv_pgd3, linestyle=':', label='PGD $\\alpha=0.05$')
ax.set_xlabel('Iteration'); ax.set_ylabel('$f$ [log]')
ax.set_title('Q3: (e) FW vs PGD: $f(x^{(t)})$')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
draw_feasible_X3(ax)
ax.contour(X1G3, X2G3, ZQ3, levels=15, linewidths=0.7, zorder=1)
ax.plot(hist_fw08_q3[:,0], hist_fw08_q3[:,1], '-', label='FW $\\beta=0.8$')
ax.plot(hist_fw095_q3[:,0], hist_fw095_q3[:,1], '--', label='FW $\\beta=0.95$')
ax.plot(hist_pgd3[:,0], hist_pgd3[:,1], ':', label='PGD')
ax.plot(*x0_q3, 's', ms=8, label='$x^{(0)}$')
ax.set_xlim(0,3.5); ax.set_ylim(0,3.5)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Q3: (e) Trajectory comparison')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('q3e_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()